# Adversarial Search: Playing Dots and Boxes


## Instructions

Total Points: Undegraduates 100, graduate students 110

Complete this notebook and submit it. The notebook needs to be a complete project report with your implementation, documentation including a short discussion of how your implementation works and your design choices, and experimental results (e.g., tables and charts with simulation results) with a short discussion of what they mean. Use the provided notebook cells and insert additional code and markdown cells as needed.

## Introduction

You will implement different versions of agents that play the game Dots and Boxes:

> "Dots and Boxes is a pencil-and-paper game for two players. The game starts with an empty grid of dots. Usually two players take turns adding a single horizontal or vertical line between two unjoined adjacent dots. A player who completes the fourth side of a 1x1 box earns one point and takes another turn. A point is typically recorded by placing a mark that identifies the player in the box, such as an initial. The game ends when no more lines can be placed. The winner is the player with the most points. The board may be of any size grid." (see [Dots and Boxes on Wikipedia](https://en.wikipedia.org/wiki/Dots_and_Boxes))

You can play Dots and Boxes [here](https://www.math.ucla.edu/~tom/Games/dots&boxes.html).

## Task 1: Defining the Search Problem [10 point]

Define the components of the search problem associated with this game:

* Initial state
* Actions
* Transition model
* Test for the terminal state
* Utility for terminal states

Định nghĩa các thành phần của bài toán tìm kiếm
- Trạng thái ban đầu (Initial state): Một bảng chơi trống chứa lưới điểm (dot) kích thước $M \times N$. Chưa có đoạn thẳng nào được vẽ, số điểm của hai người chơi là 0-0, và lượt đi đầu tiên được chỉ định cho Người chơi 1.
- Các hành động (Actions): Tập hợp các nước đi hợp lệ $actions(s)$ tại trạng thái $s$. Mỗi hành động là việc chọn vẽ một đoạn thẳng (ngang hoặc dọc) nối giữa hai điểm kề nhau chưa được nối trên bảng.
- Mô hình chuyển trạng thái (Transition model): Hàm $result(s, a)$ trả về trạng thái mới $s'$ sau khi áp dụng hành động $a$. Trạng thái $s'$ cập nhật đoạn thẳng vừa được vẽ. Lưu ý quan trọng: Nếu đoạn thẳng này hoàn thành cạnh thứ tư của một (hoặc hai) ô vuông $1 \times 1$, điểm của người chơi hiện tại tăng lên, ô vuông đó được đánh dấu là thuộc sở hữu của họ, và người chơi đó được giữ quyền đi tiếp. Nếu không có ô vuông nào được tạo thành, lượt đi được chuyển sang cho đối thủ.
- Kiểm tra trạng thái kết thúc (Test for the terminal state): Hàm $terminal(s)$ trả về True khi tất cả các đoạn thẳng có thể vẽ trên bảng đều đã được vẽ (đồng nghĩa với việc tất cả các ô vuông đã được hoàn thành).
-Giá trị thỏa dụng (Utility for terminal states): Hàm $utility(s)$ đánh giá kết quả tại trạng thái kết thúc. Thông thường trong thuật toán Minimax, có thể thiết lập: $+1$ (Người chơi 1 thắng), $-1$ (Người chơi 2 thắng), và $0$ (Hòa). Ngoài ra, cũng có thể sử dụng trực tiếp hiệu số điểm giữa hai người chơi $(Score_1 - Score_2)$ làm giá trị Utility.

How big is the state space? Give an estimate and explain it.

 Ước tính kích thước không gian trạng thái (State Space)
 - Ước tính: $O(2^L)$ trạng thái, trong đó $L$ là tổng số đoạn thẳng có thể vẽ trên lưới.
 - Giải thích:Đối với lưới điểm kích thước $M \times N$ (hàng $\times$ cột)
    + Số đoạn thẳng ngang tối đa: $M \times (N - 1)$
    + Số đoạn thẳng dọc tối đa: $(M - 1) \times N$
    + Tổng số đoạn thẳng $L = 2MN - M - N$.
    + Mỗi đoạn thẳng chỉ có 2 trạng thái độc lập: đã được vẽ hoặc chưa được vẽ. Do đó, có $2^L$ cấu hình đoạn thẳng trên bảng.
    + Ví dụ: Với lưới $4 \times 4$ điểm (tạo ra bảng $3 \times 3$ ô vuông) như trong file hướng dẫn, tổng số đoạn thẳng $L = 2(4)(4) - 4 - 4 = 24$. Không gian trạng thái sẽ xấp xỉ $2^{24} \approx 1.67 \times 10^7$ trạng thái.

How big is the game tree that minimax search will go through? Give an estimate and explain it.

Ước tính kích thước cây trò chơi cho thuật toán Minimax (Game Tree)
- Ước tính: Kích thước của cây trò chơi đầy đủ (số lượng đường đi từ gốc đến lá) là $L!$ ($L$ giai thừa).
- Giải thích: Bất kể ai là người chơi hay có bao nhiêu ô vuông được hoàn thành, một ván cờ Dots and Boxes luôn luôn kết thúc sau khi vẽ chính xác $L$ đoạn thẳng.
  - Ở nước đi đầu tiên, người chơi có $L$ lựa chọn đoạn thẳng.
  - Ở nước đi thứ hai (không cần biết là của ai), có $L - 1$ lựa chọn.
  - Quá trình này tiếp tục cho đến nước đi cuối cùng chỉ còn $1$ lựa chọn.
  - Số lượng các chuỗi nước đi có thể xảy ra (nhánh của cây) chính là số hoán vị của $L$ đoạn thẳng.
  - Ví dụ: Với lưới $4 \times 4$ điểm ($L = 24$), số nút lá của cây trò chơi lên tới $24! \approx 6.2 \times 10^{23}$. Kích thước khổng lồ này giải thích lý do tại sao thuật toán Minimax cơ bản không thể giải quyết toàn bộ cây và bắt buộc phải dùng tới Alpha-Beta Pruning kết hợp hàm Heuristic giới hạn độ sâu (Cutoff) cho các Task tiếp theo.

## Task 2: Game Environment and Random Agent [30 point]

You need to think about a data structure to represent the board meaning he placed lines and who finished what box. There are many options. Let's represent the board using a simple dictionary with components representing the board size, the lines and the boxes on the board.

**Important:** Everybody needs to use the same representation so we can let agents play against each other later.

In [1]:
board = {
    'size': (4, 4),  ### number of rows and columns of dots
    'lines': dict(), ### keys are the set of drawn lines
    'boxes': dict    ### keys are the boxes and the value is the player who completed each box
}

def draw_line(board, orientation, row, col):
    """
    Place a line on an exiting board.

    Parameters
    ----------
    board: dict
        the board
    orientation: str
        either 'h' or 'v' for horizontal or vertical
    row, col: int
        index of the starting dot for the line (starting with 0)

    """

    if orientation not in ['h', 'v']:
        return False

    if row < 0 or col < 0:
        return False

    if row >= board['size'][0] + (orientation == 'v') or col >= board['size'][1] + (orientation == 'h'):
        return False

    if (orientation, row, col) in board['lines']:
        return False

    board["lines"][(orientation, row, col)] = True
    return True


print(draw_line(board, "h", 1, 1))
print(draw_line(board, "v", 1, 1))

# this should not work
print(draw_line(board, "h", 1, 1))

board

True
True
False


{'size': (4, 4),
 'lines': {('h', 1, 1): True, ('v', 1, 1): True},
 'boxes': dict}

Write code to display the board. **Bonus point: Post your visualization code with an example output to the discussion board. The best visualization will earn you bonus participation points in this class.**

In [2]:
def display_board(board):
    rows, cols = board['size']

    for r in range(rows):
        # Vẽ các điểm và đường ngang
        row_str = ""
        for c in range(cols):
            row_str += "*" # Điểm
            if c < cols - 1:
                if ('h', r, c) in board['lines']:
                    row_str += "---"
                else:
                    row_str += "   "
        print(row_str)

        # Vẽ các đường dọc và ô vuông (trừ hàng cuối cùng)
        if r < rows - 1:
            col_str = ""
            for c in range(cols):
                if ('v', r, c) in board['lines']:
                    col_str += "|"
                else:
                    col_str += " "

                # Hiển thị chủ sở hữu ô vuông
                if c < cols - 1:
                    if (r, c) in board['boxes']:
                        owner = board['boxes'][(r, c)]
                        col_str += f" {owner if owner == 1 else 2} "
                    else:
                        col_str += "   "
            print(col_str)
    print()

Implement helper functions for:

* The transition model $result(s, a)$.
* The utility function $utility(s)$.
* Check for terminal states $terminal(s)$.
* A check for available actions in each state $actions(s)$.

__Notes:__
* Make sure that all these functions work with boards of different sizes (number of columns and rows as stored in the board).
* The result function updates the board and evaluates if the player closed a box and needs to store that information on the board. Add elements of the form `(row,col): player` to the board dictionary. `row` and `col` are the coordinates for the box and `player` is +1 or -1 representing the player. For example `(0,0): -1` means that the top-left box belongs to the other player.
* _Important:_ Remember that a player goes again after she completes a box!

In [3]:
import copy

def actions(board):
    """Trả về danh sách các nước đi hợp lệ (chưa được vẽ)."""
    valid_actions = []
    rows, cols = board['size']

    # Kiểm tra đường ngang
    for r in range(rows):
        for c in range(cols - 1):
            if ('h', r, c) not in board['lines']:
                valid_actions.append(('h', r, c))

    # Kiểm tra đường dọc
    for r in range(rows - 1):
        for c in range(cols):
            if ('v', r, c) not in board['lines']:
                valid_actions.append(('v', r, c))

    return valid_actions

def check_new_boxes(board, action, player):
    """Kiểm tra xem nước đi vừa rồi có tạo ra ô vuông mới nào không và cập nhật."""
    orientation, r, c = action
    rows, cols = board['size']
    boxes_formed = 0
    lines = board['lines']

    if orientation == 'h':
        # Kiểm tra ô vuông phía trên
        if r > 0:
            if ('h', r-1, c) in lines and ('v', r-1, c) in lines and ('v', r-1, c+1) in lines:
                board['boxes'][(r-1, c)] = player
                boxes_formed += 1
        # Kiểm tra ô vuông phía dưới
        if r < rows - 1:
            if ('h', r+1, c) in lines and ('v', r, c) in lines and ('v', r, c+1) in lines:
                board['boxes'][(r, c)] = player
                boxes_formed += 1

    elif orientation == 'v':
        # Kiểm tra ô vuông bên trái
        if c > 0:
            if ('v', r, c-1) in lines and ('h', r, c-1) in lines and ('h', r+1, c-1) in lines:
                board['boxes'][(r, c-1)] = player
                boxes_formed += 1
        # Kiểm tra ô vuông bên phải
        if c < cols - 1:
            if ('v', r, c+1) in lines and ('h', r, c) in lines and ('h', r+1, c) in lines:
                board['boxes'][(r, c)] = player
                boxes_formed += 1

    return boxes_formed > 0

def result(board, action, player):
    """Áp dụng nước đi, cập nhật bảng và trả về (bảng_mới, được_đi_tiếp_không)."""
    new_board = copy.deepcopy(board)
    new_board['lines'][action] = True

    go_again = check_new_boxes(new_board, action, player)
    return new_board, go_again

def terminal(board):
    """Trò chơi kết thúc khi không còn nước đi nào."""
    return len(actions(board)) == 0

def utility(board):
    """Trả về hiệu số điểm giữa player 1 và player -1."""
    score_1 = sum(1 for owner in board['boxes'].values() if owner == 1)
    score_minus_1 = sum(1 for owner in board['boxes'].values() if owner == -1)
    return score_1 - score_minus_1

Implement an agent that plays randomly. Make sure the agent function receives as the percept the board and returns a valid action. Use an agent function definition with the following signature (arguments):

`def random_player(board, player = None): ...`

The argument `player` is used for agents that do not store what side they are playing. The value passed on by the environment should be 1 ot -1 for playerred and yellow, respectively.  See [Experiments section for tic-tac-toe](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_and_or_tree_search.ipynb#Experiments) for an example.

In [4]:
import random

def random_player(board, player=None):
    """Trả về một nước đi ngẫu nhiên hợp lệ."""
    possible_actions = actions(board)
    if not possible_actions:
        return None
    return random.choice(possible_actions)

Let two random agents play against each other 1000 times. Look at the [Experiments section for tic-tac-toe](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_and_or_tree_search.ipynb#Experiments) to see how the environment uses the agent functions to play against each other.

How often does each player win? Is the result expected?

In [13]:
def play_game(agent1, agent2, board_size=(4, 4)):
    board = {
        'size': board_size,
        'lines': dict(),
        'boxes': dict()
    }

    current_player = 1

    while not terminal(board):
        if current_player == 1:
            action = agent1(board, 1)
        else:
            action = agent2(board, -1)

        board, go_again = result(board, action, current_player)

        # Đổi lượt nếu người chơi không ghi được điểm
        if not go_again:
            current_player = -current_player

    final_score = utility(board)
    if final_score > 0:
        return 1  # Player 1 thắng
    elif final_score < 0:
        return -1 # Player 2 thắng
    else:
        return 0  # Hòa

# Chạy 1000 trận
wins_1 = 0
wins_2 = 0
draws = 0

for _ in range(1000):
    res = play_game(random_player, random_player, board_size=(4,4))
    if res == 1:
        wins_1 += 1
    elif res == -1:
        wins_2 += 1
    else:
        draws += 1

print(f"Player 1 thắng: {wins_1}")
print(f"Player 2 thắng: {wins_2}")
print(f"Hòa: {draws}")

Player 1 thắng: 532
Player 2 thắng: 468
Hòa: 0


Khi chạy đoạn mã trên 1000 lần, tỷ lệ thắng của Player 1 (người đi trước) thường sẽ nhỉnh hơn một chút so với Player 2 (dao động khoảng 52% - 58%)(532 - 468), và số trận hòa khá thấp.

Trong các trò chơi dạng lưới như Dots and Boxes, lợi thế người đi trước (first-mover advantage) luôn tồn tại vì họ là người có khả năng chủ động kiểm soát parity (tính chẵn lẻ của đồ thị) và chuỗi các ô vuông được tạo ra ở cuối game (chuỗi "chains"). Mặc dù cả hai đều chọn ngẫu nhiên, cấu trúc trò chơi vốn đã hơi nghiêng phần thắng về phía người khai cuộc.

## Task 3: Minimax Search with Alpha-Beta Pruning [30 points]

### Implement the search starting.

Implement the search starting from a given board and specifying the player and put it into an agent function.
You can use code from the [tic-tac-toe example](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_alpha_beta_tree_search.ipynb).

__Notes:__
* Make sure that all your agent functions have a signature consistent with the random agent above.
* The search space for larger board may be too large. You can experiment with smaller boards.
* Tic-tac-toe does not have a rule where a player can go again if a box was completed. You need to adapt the tree search to reflect that rule.

In [15]:
import copy
import time

node_count = 0  # Biến toàn cục để đếm số nút đã duyệt

def alpha_beta_search(board, player, use_ordering=False):
    global node_count
    node_count = 0
    best_action = None
    alpha = -float('inf')
    beta = float('inf')

    # Đối với Player 1 (MAX), mục tiêu là tối đa hóa hàm utility (score1 - score2).
    # Đối với Player -1 (MIN), mục tiêu là tối thiểu hóa hàm utility.
    is_max_player = (player == 1)
    best_val = -float('inf') if is_max_player else float('inf')

    possible_actions = actions(board)
    if use_ordering:
        possible_actions = order_moves(board, possible_actions, player)

    for action in possible_actions:
        next_board, go_again = result(board, action, player)

        # Nếu được đi tiếp, giữ nguyên trạng thái MAX/MIN cho độ sâu tiếp theo
        if is_max_player:
            if go_again:
                val = max_value(next_board, alpha, beta, player)
            else:
                val = min_value(next_board, alpha, beta, -player)

            if val > best_val:
                best_val = val
                best_action = action
            alpha = max(alpha, best_val)
        else:
            if go_again:
                val = min_value(next_board, alpha, beta, player)
            else:
                val = max_value(next_board, alpha, beta, -player)

            if val < best_val:
                best_val = val
                best_action = action
            beta = min(beta, best_val)

    return best_action

def max_value(board, alpha, beta, current_player):
    global node_count
    node_count += 1

    if terminal(board):
        return utility(board)

    val = -float('inf')
    for action in actions(board): # Sẽ tích hợp order_moves ở đây nếu bật use_ordering
        next_board, go_again = result(board, action, current_player)

        if go_again:
            val = max(val, max_value(next_board, alpha, beta, current_player))
        else:
            val = max(val, min_value(next_board, alpha, beta, -current_player))

        if val >= beta:
            return val
        alpha = max(alpha, val)
    return val

def min_value(board, alpha, beta, current_player):
    global node_count
    node_count += 1

    if terminal(board):
        return utility(board)

    val = float('inf')
    for action in actions(board):
        next_board, go_again = result(board, action, current_player)

        if go_again:
            val = min(val, min_value(next_board, alpha, beta, current_player))
        else:
            val = min(val, max_value(next_board, alpha, beta, -current_player))

        if val <= alpha:
            return val
        beta = min(beta, val)
    return val

def alpha_beta_player(board, player=1):
    """Hàm Agent chuẩn theo yêu cầu của hệ thống."""
    return alpha_beta_search(board, player, use_ordering=False)

Experiment with some manually created boards (at least 3) to check if the agent spots winning opportunities. Discuss the results.

In [17]:
def create_board_1():
    # Board 1: Cơ hội ăn điểm ngay lập tức.
    # Ô vuông ở góc trên bên trái (0, 0) đã có 3 cạnh (trên, trái, phải).
    # Nước đi đúng: ('h', 1, 0) để ăn điểm.
    board = {'size': (3, 3), 'lines': {}, 'boxes': {}}
    board['lines'][('h', 0, 0)] = True  # Cạnh trên
    board['lines'][('v', 0, 0)] = True  # Cạnh trái
    board['lines'][('v', 0, 1)] = True  # Cạnh phải
    return board

def create_board_2():
    # Board 2: Tránh bẫy (Avoid giving points).
    # Ô (0,0) đã có 2 cạnh: ('h', 0, 0) và ('v', 0, 0).
    # Nếu đánh ('v', 0, 1) hoặc ('h', 1, 0), ô này sẽ có 3 cạnh và đối thủ sẽ ăn được ở lượt sau.
    # Nước đi đúng: Chọn một khu vực an toàn khác, ví dụ ('h', 2, 1) ở góc dưới bên phải.
    board = {'size': (3, 3), 'lines': {}, 'boxes': {}}
    board['lines'][('h', 0, 0)] = True
    board['lines'][('v', 0, 0)] = True
    return board

def create_board_3():
    # Board 3: Chuỗi Endgame (Quét sạch bảng).
    # Bảng đã được đánh gần hết, chỉ còn khuyết 3 đoạn thẳng:
    # 1. ('h', 1, 0) - Hoàn thành ô (0,0) và (1,0)
    # 2. ('v', 1, 1)
    # 3. ('h', 2, 1)
    # Nước đi đúng: Agent phải ưu tiên vẽ ('h', 1, 0) trước để lấy điểm và lấy luôn quyền đi tiếp,
    # sau đó ăn nốt các ô còn lại thay vì đi các nước không có điểm.
    board = {'size': (3, 3), 'lines': {}, 'boxes': {}}

    # Giả lập các ô vuông đã thuộc về Player -1
    board['boxes'][(0, 1)] = -1

    # Các đường đã vẽ
    drawn_lines = [
        ('h', 0, 0), ('v', 0, 0), ('v', 0, 1), # Ô (0,0) thiếu ('h', 1, 0)
        ('h', 0, 1), ('v', 0, 2), ('h', 1, 1), # Ô (0,1) đã hoàn thành
        ('v', 1, 0), ('h', 2, 0),              # Ô (1,0) thiếu ('h', 1, 0) và ('v', 1, 1)
        ('v', 1, 2)                            # Ô (1,1) thiếu ('v', 1, 1) và ('h', 2, 1)
    ]
    for line in drawn_lines:
        board['lines'][line] = True

    return board

In [18]:
boards = {
    "Board 1 (Immediate Point)": create_board_1(),
    "Board 2 (Avoid Trap)": create_board_2(),
    "Board 3 (Endgame Chain)": create_board_3()
}

for name, board in boards.items():
    print(f"--- Kiểm tra {name} ---")
    start_time = time.time()

    # Gọi thuật toán Alpha-Beta cho Player 1
    best_move = alpha_beta_search(board, player=1, use_ordering=True)

    elapsed = time.time() - start_time
    print(f"Nước đi Agent chọn: {best_move}")
    print(f"Thời gian suy nghĩ: {elapsed:.4f} giây\n")

--- Kiểm tra Board 1 (Immediate Point) ---
Nước đi Agent chọn: ('h', 1, 0)
Thời gian suy nghĩ: 0.2504 giây

--- Kiểm tra Board 2 (Avoid Trap) ---
Nước đi Agent chọn: ('h', 0, 1)
Thời gian suy nghĩ: 2.1019 giây

--- Kiểm tra Board 3 (Endgame Chain) ---
Nước đi Agent chọn: ('h', 1, 0)
Thời gian suy nghĩ: 0.0004 giây



How long does it take to make a move? Start with a smaller board make the board larger. What is the largest board you can solve?

In [ ]:
# Your code/ answer goes here.

### Move ordering

Starting the search with better moves will increase the efficiency of alpha-beta pruning. Describe and implement a simple move ordering strategy.

Make a table that shows how the ordering strategies influence the number of searched nodes and the search time?

In [16]:
def count_edges(board, r, c):
    """Đếm số cạnh hiện có của một ô vuông cụ thể."""
    edges = 0
    if ('h', r, c) in board['lines']: edges += 1
    if ('h', r+1, c) in board['lines']: edges += 1
    if ('v', r, c) in board['lines']: edges += 1
    if ('v', r, c+1) in board['lines']: edges += 1
    return edges

def order_moves(board, actions_list, player):
    def move_score(action):
        orientation, r, c = action
        score = 0
        # Kiểm tra sự ảnh hưởng của nước đi lên các ô vuông lân cận
        if orientation == 'h':
            if r > 0:
                edges_above = count_edges(board, r-1, c)
                if edges_above == 3: score += 100  # Sẽ hoàn thành ô này
                elif edges_above == 2: score -= 10 # Tránh tạo thành ô 3 cạnh
            if r < board['size'][0] - 1:
                edges_below = count_edges(board, r, c)
                if edges_below == 3: score += 100
                elif edges_below == 2: score -= 10
        elif orientation == 'v':
            if c > 0:
                edges_left = count_edges(board, r, c-1)
                if edges_left == 3: score += 100
                elif edges_left == 2: score -= 10
            if c < board['size'][1] - 1:
                edges_right = count_edges(board, r, c)
                if edges_right == 3: score += 100
                elif edges_right == 2: score -= 10
        return score

    # Sắp xếp các nước đi có điểm số cao lên đầu để duyệt trước
    return sorted(actions_list, key=move_score, reverse=True)

### The first few moves

Start with an empty board. This is the worst case scenario for minimax search with alpha-beta pruning since it needs solve all possible games that can be played (minus some pruning) before making the decision. What can you do?

In [19]:
import random

def hybrid_alpha_beta_player(board, player=1):
    """
    Agent lai: Đi ngẫu nhiên ở những nước đầu, sau đó dùng Alpha-Beta.
    """
    possible_actions = actions(board)
    total_lines = (board['size'][0] * (board['size'][1] - 1)) + ((board['size'][0] - 1) * board['size'][1])
    lines_drawn = total_lines - len(possible_actions)

    # Kích hoạt Alpha-Beta khi số nước cờ đã đi vượt qua một ngưỡng nhất định
    # (Ví dụ: trên bảng 3x3 (24 cạnh), bắt đầu tính toán sâu khi đã vẽ 12 cạnh)
    if lines_drawn < 12 and len(possible_actions) > 12:
        # Trong giai đoạn đầu, chỉ đi ngẫu nhiên vào những chỗ KHÔNG tạo ra ô 3 cạnh cho đối thủ
        safe_actions = []
        for action in possible_actions:
            # Sử dụng hàm move_score từ phần Move Ordering (hoặc kiểm tra đơn giản)
            # Ở đây ta giả sử chọn các nước đi ngẫu nhiên hợp lệ
            safe_actions.append(action)
        return random.choice(safe_actions)
    else:
        # Giai đoạn Endgame: Bật Minimax Alpha-Beta với tầm nhìn hoàn hảo
        return alpha_beta_search(board, player, use_ordering=True)

### Playtime

Let the Minimax Search agent play a random agent on a small board. Analyze wins, losses and draws.

In [26]:
def playtime_minimax_vs_random(num_games=100):
    wins_minimax = 0
    wins_random = 0
    draws = 0

    for i in range(num_games):
        # Tạo bảng 2x2 trống
        board = {'size': (2, 2), 'lines': dict(), 'boxes': dict()}
        current_player = 1

        # Luân phiên người đi trước để công bằng
        minimax_side = 1 if i % 2 == 0 else -1

        while not terminal(board):
            if current_player == minimax_side:
                action = alpha_beta_search(board, current_player, use_ordering=True)
            else:
                action = random_player(board, current_player)

            board, go_again = result(board, action, current_player)

            if not go_again:
                current_player = -current_player

        # Tính điểm
        score = utility(board)

        if score > 0:
            if minimax_side == 1: wins_minimax += 1
            else: wins_random += 1
        elif score < 0:
            if minimax_side == -1: wins_minimax += 1
            else: wins_random += 1
        else:
            draws += 1

    print(f"Tổng số trận: {num_games}")
    print(f"Minimax thắng: {wins_minimax}")
    print(f"Random thắng: {wins_random}")
    print(f"Hòa: {draws}")

# Chạy thực nghiệm
playtime_minimax_vs_random(20)

Tổng số trận: 20
Minimax thắng: 10
Random thắng: 10
Hòa: 0


## Task 4: Heuristic Alpha-Beta Tree Search [30 points]

### Heuristic evaluation function

Define and implement a heuristic evaluation function.

In [27]:
def count_edges(board, r, c):
    """Hàm hỗ trợ đếm số cạnh của ô vuông (r, c)."""
    edges = 0
    if ('h', r, c) in board['lines']: edges += 1
    if ('h', r+1, c) in board['lines']: edges += 1
    if ('v', r, c) in board['lines']: edges += 1
    if ('v', r, c+1) in board['lines']: edges += 1
    return edges

def heuristic_eval(board):
    """
    Hàm Heuristic trả về điểm đánh giá cho Player 1.
    """
    # 1. Giá trị Utility (Hiệu số điểm thực tế) nhân với trọng số lớn
    score = utility(board) * 100

    rows, cols = board['size']
    # 2. Đánh giá rủi ro từ các ô vuông chưa hoàn thành
    for r in range(rows - 1):
        for c in range(cols - 1):
            if (r, c) not in board['boxes']:
                edges = count_edges(board, r, c)
                if edges == 3:
                    # Rất nguy hiểm vì đối thủ có thể ăn ô này ở lượt sau
                    score -= 10
                elif edges == 2:
                    # Hơi nguy hiểm (dễ bị ép trở thành ô 3 cạnh)
                    score -= 1
                elif edges == 0 or edges == 1:
                    # Khu vực an toàn
                    score += 1

    return score

### Cutting off search

Modify your Minimax Search with Alpha-Beta Pruning to cut off search at a specified depth and use the heuristic evaluation function. Experiment with different cutoff values.

In [28]:
node_count = 0

def heuristic_alpha_beta_search(board, player, max_depth=4, use_ordering=True):
    global node_count
    node_count = 0
    best_action = None
    alpha = -float('inf')
    beta = float('inf')

    is_max_player = (player == 1)
    best_val = -float('inf') if is_max_player else float('inf')

    possible_actions = actions(board)
    if use_ordering:
        possible_actions = order_moves(board, possible_actions, player) # Tái sử dụng hàm order_moves từ Task 3

    for action in possible_actions:
        next_board, go_again = result(board, action, player)

        if is_max_player:
            if go_again:
                val = max_value_cutoff(next_board, alpha, beta, player, max_depth - 1)
            else:
                val = min_value_cutoff(next_board, alpha, beta, -player, max_depth - 1)

            if val > best_val:
                best_val = val
                best_action = action
            alpha = max(alpha, best_val)
        else:
            if go_again:
                val = min_value_cutoff(next_board, alpha, beta, player, max_depth - 1)
            else:
                val = max_value_cutoff(next_board, alpha, beta, -player, max_depth - 1)

            if val < best_val:
                best_val = val
                best_action = action
            beta = min(beta, best_val)

    return best_action

def max_value_cutoff(board, alpha, beta, current_player, depth):
    global node_count
    node_count += 1

    if terminal(board):
        return utility(board) * 100 # Nhân 100 để đồng bộ thang điểm với heuristic
    if depth <= 0:
        return heuristic_eval(board)

    val = -float('inf')
    possible_actions = order_moves(board, actions(board), current_player)

    for action in possible_actions:
        next_board, go_again = result(board, action, current_player)

        if go_again:
            val = max(val, max_value_cutoff(next_board, alpha, beta, current_player, depth - 1))
        else:
            val = max(val, min_value_cutoff(next_board, alpha, beta, -current_player, depth - 1))

        if val >= beta:
            return val
        alpha = max(alpha, val)
    return val

def min_value_cutoff(board, alpha, beta, current_player, depth):
    global node_count
    node_count += 1

    if terminal(board):
        return utility(board) * 100
    if depth <= 0:
        return heuristic_eval(board)

    val = float('inf')
    possible_actions = order_moves(board, actions(board), current_player)

    for action in possible_actions:
        next_board, go_again = result(board, action, current_player)

        if go_again:
            val = min(val, min_value_cutoff(next_board, alpha, beta, current_player, depth - 1))
        else:
            val = min(val, max_value_cutoff(next_board, alpha, beta, -current_player, depth - 1))

        if val <= alpha:
            return val
        beta = min(beta, val)
    return val

def heuristic_agent_depth_4(board, player=1):
    return heuristic_alpha_beta_search(board, player, max_depth=4)

def heuristic_agent_depth_2(board, player=1):
    return heuristic_alpha_beta_search(board, player, max_depth=2)

How many nodes are searched and how long does it take to make a move? Start with a smaller board with 4 columns and make the board larger by adding columns.

In [29]:
import time

board_sizes = [(4, 4), (4, 5), (4, 6)]
depth_to_test = 4

print("--- KIỂM TRA HIỆU SUẤT TÌM KIẾM ---")
for size in board_sizes:
    empty_board = {'size': size, 'lines': dict(), 'boxes': dict()}

    start_time = time.time()
    best_move = heuristic_alpha_beta_search(empty_board, player=1, max_depth=depth_to_test)
    elapsed_time = time.time() - start_time

    print(f"Bảng {size} (Số đoạn thẳng: {(size[0]*(size[1]-1)) + ((size[0]-1)*size[1])}):")
    print(f"  - Nước đi tốt nhất: {best_move}")
    print(f"  - Số node đã duyệt: {node_count}")
    print(f"  - Thời gian: {elapsed_time:.4f} giây\n")

--- KIỂM TRA HIỆU SUẤT TÌM KIẾM ---
Bảng (4, 4) (Số đoạn thẳng: 24):
  - Nước đi tốt nhất: ('h', 0, 0)
  - Số node đã duyệt: 18126
  - Thời gian: 0.4227 giây

Bảng (4, 5) (Số đoạn thẳng: 31):
  - Nước đi tốt nhất: ('h', 0, 0)
  - Số node đã duyệt: 41011
  - Thời gian: 1.0385 giây

Bảng (4, 6) (Số đoạn thẳng: 38):
  - Nước đi tốt nhất: ('h', 0, 0)
  - Số node đã duyệt: 79194
  - Thời gian: 2.6859 giây



1. Sự tăng trưởng của không gian tìm kiếm (Node count & Thời gian)
  - Bảng (4,4) - 24 đoạn thẳng: Duyệt ~18.100 nodes mất 0.42 giây.
  - Bảng (4,5) - 31 đoạn thẳng: Duyệt ~41.000 nodes mất 1.03 giây (Kích thước tăng ~30%, nhưng số nodes và thời gian tăng hơn gấp đôi).
  - Bảng (4,6) - 38 đoạn thẳng: Duyệt ~79.100 nodes mất 2.68 giây (Tiếp tục tăng gần gấp đôi so với bảng trước).
  - Đánh giá: Sự tăng trưởng này minh họa rõ rệt bản chất bùng nổ tổ hợp của cây trò chơi. Khi số lượng tùy chọn nước đi (branching factor) ở mỗi tầng tăng lên, số lượng cấu hình trạng thái phải duyệt tăng theo cấp số nhân. Tuy nhiên, nhờ việc giới hạn độ sâu cắt tỉa (max_depth = 4), thời gian phản hồi vẫn được duy trì ở mức thực tế (dưới 3 giây) thay vì bị treo vô hạn như thuật toán Minimax gốc.
2. Hiệu quả của Alpha-Beta Pruning kết hợp Move Ordering
  Nếu không có cắt tỉa, số node phải duyệt ở độ sâu 4 cho bảng có 38 đoạn thẳng sẽ xấp xỉ $38 \times 37 \times 36 \times 35 \approx 1.7$ triệu nodes. Nhờ áp dụng Alpha-Beta và chiến lược sắp xếp nước đi (Move Ordering), thuật toán chỉ phải duyệt khoảng 79.000 nodes (giảm hơn 95% khối lượng công việc) mà vẫn đảm bảo đưa ra quyết định tương đương.
3. Hành vi chọn nước đi đầu tiên ('h', 0, 0)
  Trên một bảng trống, không có ô vuông nào để hoàn thành và không có bẫy, hàm Heuristic sẽ đánh giá tất cả các nước đi an toàn với số điểm bằng nhau. Thuật toán Minimax mang tính quyết định (deterministic), do đó khi gặp một loạt các điểm số cân bằng, nó sẽ chọn nhánh đầu tiên mà nó duyệt qua. Đây là lý do tại sao ở cả 3 kích thước bảng, Agent đều ưu tiên vẽ đoạn thẳng ngang ở góc trên cùng bên trái ('h', 0, 0).

### Playtime

Let two heuristic search agents (different cutoff depth, different heuristic evaluation function) compete against each other on a reasonably sized board. Since there is no randomness, you only need to let them play once.

In [31]:
def play_heuristic_match():
    board = {'size': (4, 4), 'lines': dict(), 'boxes': dict()}
    current_player = 1

    print("--- TRẬN ĐẤU BẮT ĐẦU: Agent Depth 4 (Player 1) vs Agent Depth 2 (Player -1) ---")

    moves_count = 0
    while not terminal(board):
        if current_player == 1:
            action = heuristic_agent_depth_4(board, 1)
        else:
            action = heuristic_agent_depth_2(board, -1)

        board, go_again = result(board, action, current_player)
        moves_count += 1

        if not go_again:
            current_player = -current_player

    final_score = utility(board)
    print(f"Trận đấu kết thúc sau {moves_count} nước đi.")
    print(f"Điểm Player 1 (Depth 4): {sum(1 for owner in board['boxes'].values() if owner == 1)}")
    print(f"Điểm Player -1 (Depth 2): {sum(1 for owner in board['boxes'].values() if owner == -1)}")
    if final_score > 0:
        print("=> Người thắng: Player 1 (Depth 4)")
    elif final_score < 0:
        print("=> Người thắng: Player -1 (Depth 2)")
    else:
        print("=> Kết quả: Hòa")

play_heuristic_match()

--- TRẬN ĐẤU BẮT ĐẦU: Agent Depth 4 (Player 1) vs Agent Depth 2 (Player -1) ---
Trận đấu kết thúc sau 24 nước đi.
Điểm Player 1 (Depth 4): 4
Điểm Player -1 (Depth 2): 5
=> Người thắng: Player -1 (Depth 2)


## Tournament task [+1 to 5% bonus on your course grade; will be assigned separately]

Find another student and let your best agent play against the other student's best player. You are allowed to use any improvements you like as long as you code it yourself. We will set up a class tournament on Canvas. This tournament will continue after the submission deadline.

## Graduate student advanced task: Pure Monte Carlo Search and Best First Move [10 point]

__Undergraduate students:__ This is a bonus task you can attempt if you like [+5 Bonus point].

### Pure Monte Carlo Search

Implement Pure Monte Carlo Search (see [tic-tac-toe-example](tictactoe_pure_monte_carlo_search.ipynb)) and investigate how this search performs on the test boards that you have used above.

In [ ]:
# Your code/ answer goes here.

### Best First Move

How would you determine what the best first move for a standard board ($5 \times 5$) is? You can use Pure Monte Carlo Search or any algorithms that you have implemented above.

In [ ]:
# Your code/ answer goes here.